# Climate-data processing with CDO and NCO
### Yiwen Li · Fresh Eyes on CMIP

Compute temperature anomalies, an Arctic time series and climate stripes from
**ACCESS-CM2 historical + SSP5-8.5 monthly temperature**. Follow the CDO and NCO
commands step by step, then compare their results with an independent Python calculation.

The shared scientific definitions follow
[Evgenia Galytska's ESMValTool recipe](https://github.com/egalytska/Fresh_eyes/blob/040b9c7962355b1f8fa1312b7af867b458102b85/ESMValTool/recipe_temperature_anomalies.yml).
The [original Python workshop by Abhnil](https://github.com/abhnil/Fresh_Eyes_on_CMIP_Data_Workshop)
provides the broader workshop context.

Run the cells in order with **Shift+Enter**, including the collapsed cells.

## 01 · Data

| Item | Dataset |
|---|---|
| Model variable | ACCESS-CM2 monthly near-surface air temperature (`tas`, Kelvin) |
| Historical file | 1850–2014 |
| Future file | SSP5-8.5, 2015–2100 |
| Analysis period | 1979–2024: 552 months |
| Monthly reference | 1981–2010 |

Run the next cell to select the data and create a new results folder.

In [ ]:
# Select the data and create this run
from pathlib import Path
import os, sys, subprocess, json, uuid, shlex
from datetime import datetime, timezone
from IPython.display import display, Image
from IPython.core.magic import no_var_expand

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "scripts/run_python_updated.py").is_file()), None)
assert ROOT is not None, "Open this notebook from the workshop repository."
os.chdir(ROOT)

# The seminar uses real ACCESS-CM2. Synthetic mode is an explicit software-test option.
DATA_MODE = os.environ.get("WORKSHOP_MODE", "real")  # "real" or "synthetic"
END_YEAR = 2024
if os.environ.get("WORKSHOP_END_YEAR", "2024") != "2024":
    raise ValueError("This command-by-command lesson uses explicit 1979–2024 dates. Use Run_Updated_Demo.command --end-year 2100 for the extended batch calculation.")
assert DATA_MODE in {"synthetic", "real"}
assert END_YEAR >= 2024
os.environ["OMP_NUM_THREADS"] = "1"
if DATA_MODE == "synthetic":
    fixture = ROOT / "data/updated_fixture"
    subprocess.run([sys.executable, str(ROOT / "scripts/generate_updated_fixture.py"), str(fixture)], check=True)
    HIST, FUTURE = fixture / "historical.nc", fixture / "ssp585.nc"
else:
    directory = ROOT / "data/access_cm2"
    HIST = Path(os.environ.get("WORKSHOP_HISTORICAL", str(directory / "tas_Amon_ACCESS-CM2_historical_r1i1p1f1_gn_185001-201412.nc")))
    FUTURE = Path(os.environ.get("WORKSHOP_FUTURE", str(directory / "tas_Amon_ACCESS-CM2_ssp585_r1i1p1f1_gn_201501-210012.nc")))
    missing = [str(p) for p in (HIST, FUTURE) if not p.is_file()]
    if missing:
        raise FileNotFoundError("Real-data mode requires both raw files. No synthetic substitution is allowed. Missing: " + "; ".join(missing) + ". See docs/UPDATED_DEMO.md.")

HIST, FUTURE = HIST.resolve(), FUTURE.resolve()
RUN = ROOT / "outputs" / ("updated_" + DATA_MODE + "_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:6])
RUN.mkdir(parents=True)
for directory in ("data", "cdo", "nco", "exercises"):
    (RUN / directory).mkdir()
(RUN / "data/historical.nc").symlink_to(HIST.resolve())
(RUN / "data/ssp585.nc").symlink_to(FUTURE.resolve())
os.chdir(RUN)

# Execute each visible command directly; no shell expansion or script wrapper.
# A failed command raises an exception, so Run All cannot silently continue.
def run_command(command):
    result = subprocess.run(shlex.split(command), stdin=subprocess.DEVNULL,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(result.stdout, end="")
    result.check_returncode()
ip = get_ipython()
ip.automagic = True
def register_tool(tool):
    # Preserve NCO array literals and expressions exactly as written.
    @no_var_expand
    def execute(arguments):
        run_command(tool + " " + arguments)
    ip.register_magic_function(execute, magic_kind="line", magic_name=tool)
for tool in ("cdo", "ncap2", "ncatted", "ncecat", "ncks", "ncra", "ncrcat", "ncrename", "ncwa"):
    register_tool(tool)
print("DATA MODE:", "SYNTHETIC SOFTWARE TEST — NOT ACCESS-CM2" if DATA_MODE == "synthetic" else "REAL ACCESS-CM2: historical + SSP5-8.5")
print("Analysis:", f"1979–{END_YEAR}", "| Monthly reference: 1981–2010")
print("Outputs:", RUN.relative_to(ROOT))

## 02 · The calculation we are comparing

| Step | Definition used by every route |
|---|---|
| Input | ACCESS-CM2 monthly `tas` in Kelvin, historical + SSP5-8.5; or explicitly labeled synthetic test data |
| Main period | January 1979–December 2024: **552 months, 46 annual values** |
| Reference | January 1981–December 2010: 30 values for each calendar month |
| Monthly anomaly | Temperature minus the matching calendar-month reference field |
| Mean anomaly map | **Month-duration weighted** mean over the analysis period |
| Arctic mean | Latitude centres 65–90°N, all longitudes, **equal grid-point weights** |
| Annual stripes | **Equal weight for the 12 monthly samples** in each year |
| Comparison | Maximum absolute difference ≤ 0.0003°C, relative tolerance 0; coordinates and calendar labels agree |

The map weights each monthly field by its duration. The annual series gives
each month equal weight. Keeping these definitions the same across tools makes
the numerical comparison meaningful.

In [ ]:
# Check data coverage and tool versions
_ = subprocess.run([sys.executable, str(ROOT / "scripts/check_updated_inputs.py"), str(HIST), str(FUTURE), "--end-year", str(END_YEAR)], check=True)
for command in (["cdo", "-V"], ["ncks", "--version"]):
    result = subprocess.run(command, check=True, text=True, capture_output=True)
    print((result.stdout + result.stderr).splitlines()[0])
print("Python", sys.version.split()[0])

## 03 · Reference calculation

The Python reference uses the same raw inputs and definitions. CDO and NCO
will each process the original files independently. At the end, we will compare
five products from each route.

In [ ]:
# Calculate the independent reference
_ = subprocess.run([sys.executable, str(ROOT / "scripts/run_python_updated.py"), str(HIST), str(FUTURE), str(RUN / "python"), "--end-year", str(END_YEAR)], check=True, capture_output=True, text=True)
print(f"Reference ready: five products for 1979–{END_YEAR}.")

## 04 · CDO: from monthly temperature to annual stripes

CDO operators describe the operation applied to a NetCDF file. Most steps below
write one new file, which becomes the input to the next step:
`cdo operation input.nc output.nc`.

Run **one command**, then run the next cell to **inspect its result**.
Read the output before moving on. Use **Shift+Enter** for each cell.

Run each command exactly as shown. You can use the same command text in a terminal.

- `data/historical.nc` and `data/ssp585.nc` are the two input files.
- `cdo/` and `nco/` hold the files created by each tool.
- Dates and filenames are written directly in every command.

All commands run in the results directory printed in the Data section.
To repeat the calculation, start again from the first cell to create a new
results directory. Previous results are retained.

### 04.01 · Select historical temperature

Keep near-surface air temperature from the historical file. The inspection should show the variable name `tas`.

In [ ]:
cdo selname,tas data/historical.nc cdo/historical_tas.nc

**Inspect the result**

In [ ]:
cdo showname cdo/historical_tas.nc

### 04.02 · Select 1979–2014

Keep the historical part of our analysis. The inspection should report 432 monthly records.

In [ ]:
cdo selyear,1979/2014 cdo/historical_tas.nc cdo/historical.nc

**Inspect the result**

In [ ]:
cdo ntime cdo/historical.nc

### 04.03 · Select SSP585 temperature

Keep `tas` from the SSP5-8.5 scenario file, using the same model member and grid. Check that its variable is also `tas`.

In [ ]:
cdo selname,tas data/ssp585.nc cdo/ssp585_tas.nc

**Inspect the result**

In [ ]:
cdo showname cdo/ssp585_tas.nc

### 04.04 · Select 2015–2024

Keep the scenario part needed for this analysis. The inspection should report 120 monthly records.

In [ ]:
cdo selyear,2015/2024 cdo/ssp585_tas.nc cdo/ssp585.nc

**Inspect the result**

In [ ]:
cdo ntime cdo/ssp585.nc

### 04.05 · Join the experiments

Join the historical and scenario records in time order: 552 months from 1979 through 2024. The single `-b F64` option saves double precision before calculation; subsequent files inherit it.

In [ ]:
cdo -b F64 mergetime cdo/historical.nc cdo/ssp585.nc cdo/merged.nc

**Inspect the result**

In [ ]:
cdo ntime cdo/merged.nc

### 04.06 · Convert the temperature values

Subtract 273.15 to convert Kelvin values to Celsius. The inspection summarizes the first month's minimum, mean, and maximum; `-seltimestep,1` restricts this check to one month.

In [ ]:
cdo subc,273.15 cdo/merged.nc cdo/celsius_raw.nc

**Inspect the result**

In [ ]:
cdo infon -seltimestep,1 cdo/celsius_raw.nc

### 04.07 · Update the unit label

The values have been converted; now label them `degC`. Check the unit. Changing this label alone would not convert values.

In [ ]:
cdo setunit,degC cdo/celsius_raw.nc cdo/celsius.nc

**Inspect the result**

In [ ]:
cdo showunit cdo/celsius.nc

### 04.08 · Select the 1981–2010 reference period

Keep the fixed 30-year reference period. Check that it contains 360 monthly records.

In [ ]:
cdo selyear,1981/2010 cdo/celsius.nc cdo/baseline.nc

**Inspect the result**

In [ ]:
cdo ntime cdo/baseline.nc

### 04.09 · Calculate twelve reference months

Average the 30 Januaries, the 30 Februaries, and so on, giving each matching month equal weight. Check that the output contains calendar months 1 through 12.

In [ ]:
cdo ymonmean cdo/baseline.nc cdo/climatology.nc

**Inspect the result**

In [ ]:
cdo showmon cdo/climatology.nc

### 04.10 · Subtract the matching reference month

Subtract reference January from each January, reference February from each February, and so on. Inspect the first month's anomaly range in degrees Celsius; positive values are warmer than its reference.

In [ ]:
cdo ymonsub cdo/celsius.nc cdo/climatology.nc cdo/anomaly_raw.nc

**Inspect the result**

In [ ]:
cdo infon -seltimestep,1 cdo/anomaly_raw.nc

### 04.11 · Name the anomaly variable

Rename `tas` to `tas_anom` without changing the values. Check the new variable name in the monthly anomaly product.

In [ ]:
cdo chname,tas,tas_anom cdo/anomaly_raw.nc cdo/anomaly.nc

**Inspect the result**

In [ ]:
cdo showname cdo/anomaly.nc

### 04.12 · Multiply anomalies by month length

The map uses duration weights: sum(anomaly × days) / sum(days). First multiply each anomaly by its month's number of days. The first-month summary is January's anomaly multiplied by 31.

In [ ]:
cdo muldpm cdo/anomaly.nc cdo/weighted.nc

**Inspect the result**

In [ ]:
cdo infon -seltimestep,1 cdo/weighted.nc

### 04.13 · Sum the weighted anomalies

Sum over all months at each grid point to obtain the map's numerator. The inspection now summarizes one accumulated field.

In [ ]:
cdo timsum cdo/weighted.nc cdo/weighted_sum.nc

**Inspect the result**

In [ ]:
cdo infon cdo/weighted_sum.nc

### 04.14 · Create a field of ones

Create one at every grid point and month for the denominator. The first-month minimum, mean, and maximum should all be 1. The input data have valid data throughout and complete monthly bounds.

In [ ]:
cdo expr,'tas_anom=1;' cdo/anomaly.nc cdo/ones.nc

**Inspect the result**

In [ ]:
cdo infon -seltimestep,1 cdo/ones.nc

### 04.15 · Count the days in each month

Multiply the ones by month length using the file's calendar, including leap years. For January 1979, all three summary values should be 31.

In [ ]:
cdo muldpm cdo/ones.nc cdo/days.nc

**Inspect the result**

In [ ]:
cdo infon -seltimestep,1 cdo/days.nc

### 04.16 · Sum the calendar days

Sum the month lengths to obtain the denominator. For 1979–2024, the summary should show 16,802 days everywhere.

In [ ]:
cdo timsum cdo/days.nc cdo/weight_sum.nc

**Inspect the result**

In [ ]:
cdo infon cdo/weight_sum.nc

### 04.17 · Calculate the duration-weighted map

Divide the weighted anomaly sum by the day count. Inspect the minimum, mean, and maximum of the resulting anomaly map in degrees Celsius. This preserves the upstream duration weighting.

In [ ]:
cdo div cdo/weighted_sum.nc cdo/weight_sum.nc cdo/map_mean.nc

**Inspect the result**

In [ ]:
cdo infon cdo/map_mean.nc

### 04.18 · Select the Arctic

Keep all longitudes and grid-point latitudes from 65°N to 90°N. The inspection reports how many grid points remain in this region.

In [ ]:
cdo sellonlatbox,0,360,65,90 cdo/anomaly.nc cdo/arctic.nc

**Inspect the result**

In [ ]:
cdo ngridpoints cdo/arctic.nc

### 04.19 · Calculate the monthly Arctic mean

Give every selected grid point equal weight. On this complete rectangular grid, this matches the recipe's unweighted longitude mean followed by its unweighted latitude mean. Inspect the first month's single regional anomaly.

In [ ]:
cdo fldmean,weights=false cdo/arctic.nc cdo/arctic_monthly.nc

**Inspect the result**

In [ ]:
cdo outputtab,date,value -seltimestep,1 cdo/arctic_monthly.nc

### 04.20 · Calculate the annual Arctic mean

Average each year's twelve monthly samples with equal weights, matching the upstream annual definition. Check that there are 46 annual values. The map above uses month-length weights; this annual series uses equal month weights.

In [ ]:
cdo yearmean cdo/arctic_monthly.nc cdo/arctic_annual.nc

**Inspect the result**

In [ ]:
cdo ntime cdo/arctic_annual.nc

## 05 · NCO: the same calculation from the original data

NCO uses separate commands for selecting, concatenating, calculating and
averaging. Follow the filenames from one step to the next. This route reads the
two original files and does not use the CDO results.

### 05.01 · Select historical months

Select tas for January 1979–December 2014. -v selects the variable; -d selects dates. Its coordinates and time bounds are included automatically. The input already has an expandable time dimension. The metadata shows 432 monthly records.

In [ ]:
ncks -v tas -d time,1979-01-01,2014-12-31 data/historical.nc nco/historical.nc

**Inspect the result**

In [ ]:
ncks -m -C -v tas nco/historical.nc

### 05.02 · Select SSP5-8.5 months

Select January 2015–December 2024 from the raw SSP5-8.5 file. The result contains 120 monthly records on the same grid.

In [ ]:
ncks -v tas -d time,2015-01-01,2024-12-31 data/ssp585.nc nco/ssp585.nc

**Inspect the result**

In [ ]:
ncks -m -C -v tas nco/ssp585.nc

### 05.03 · Join historical and SSP5-8.5

ncrcat joins the record dimensions in the listed order. No averaging occurs. The output contains 552 consecutive months, 1979–2024.

In [ ]:
ncrcat nco/historical.nc nco/ssp585.nc nco/merged.nc

**Inspect the result**

In [ ]:
ncks -m -C -v tas nco/merged.nc

### 05.04 · Convert Kelvin to Celsius

ncap2 -s evaluates an arithmetic expression. double(tas) retains double precision; subtracting 273.15 converts Kelvin to Celsius. The query shows the first month at one grid point.

In [ ]:
ncap2 -s 'tas=double(tas)-273.15;tas@units="degC";' nco/merged.nc nco/celsius.nc

**Inspect the result**

In [ ]:
ncks -H -C -v tas -d time,0 -d lat,0 -d lon,0 nco/celsius.nc

### 05.05 · Keep the temperature attributes consistent

ncatted -a edits attributes in place. The two missing-value attributes use the same double value. Remove the link to an unavailable external area file. These edits leave valid temperatures unchanged.

In [ ]:
ncatted -a _FillValue,tas,o,d,1.0e20 -a missing_value,tas,o,d,1.0e20 -a cell_measures,tas,d,, nco/celsius.nc

**Inspect the result**

In [ ]:
ncks -m -C -v tas nco/celsius.nc

### 05.06 · Select the reference period

Select the 360 monthly fields from January 1981 to December 2010. These records define twelve separate calendar-month reference fields.

In [ ]:
ncks -d time,1981-01-01,2010-12-31 nco/celsius.nc nco/baseline.nc

**Inspect the result**

In [ ]:
ncks -m -C -v tas nco/baseline.nc

### 05.07 · Calculate the January reference

NCO indices start at zero. The slice time,0,359,12 selects indices 0, 12, 24, …, 348: the 30 January fields in 1981–2010. ncwa -a time takes their arithmetic mean and removes the time dimension. The query shows one grid-point value.

In [ ]:
ncwa -a time -d time,0,359,12 nco/baseline.nc nco/clim_01.nc

**Inspect the result**

In [ ]:
ncks -H -C -v tas -d lat,0 -d lon,0 nco/clim_01.nc

### 05.08 · Calculate the February reference

NCO indices start at zero. The slice time,1,359,12 selects indices 1, 13, 25, …, 349: the 30 February fields in 1981–2010. ncwa -a time takes their arithmetic mean and removes the time dimension. The query shows one grid-point value.

In [ ]:
ncwa -a time -d time,1,359,12 nco/baseline.nc nco/clim_02.nc

**Inspect the result**

In [ ]:
ncks -H -C -v tas -d lat,0 -d lon,0 nco/clim_02.nc

### 05.09 · Calculate the March reference

NCO indices start at zero. The slice time,2,359,12 selects indices 2, 14, 26, …, 350: the 30 March fields in 1981–2010. ncwa -a time takes their arithmetic mean and removes the time dimension. The query shows one grid-point value.

In [ ]:
ncwa -a time -d time,2,359,12 nco/baseline.nc nco/clim_03.nc

**Inspect the result**

In [ ]:
ncks -H -C -v tas -d lat,0 -d lon,0 nco/clim_03.nc

### 05.10 · Calculate the April reference

NCO indices start at zero. The slice time,3,359,12 selects indices 3, 15, 27, …, 351: the 30 April fields in 1981–2010. ncwa -a time takes their arithmetic mean and removes the time dimension. The query shows one grid-point value.

In [ ]:
ncwa -a time -d time,3,359,12 nco/baseline.nc nco/clim_04.nc

**Inspect the result**

In [ ]:
ncks -H -C -v tas -d lat,0 -d lon,0 nco/clim_04.nc

### 05.11 · Calculate the May reference

NCO indices start at zero. The slice time,4,359,12 selects indices 4, 16, 28, …, 352: the 30 May fields in 1981–2010. ncwa -a time takes their arithmetic mean and removes the time dimension. The query shows one grid-point value.

In [ ]:
ncwa -a time -d time,4,359,12 nco/baseline.nc nco/clim_05.nc

**Inspect the result**

In [ ]:
ncks -H -C -v tas -d lat,0 -d lon,0 nco/clim_05.nc

### 05.12 · Calculate the June reference

NCO indices start at zero. The slice time,5,359,12 selects indices 5, 17, 29, …, 353: the 30 June fields in 1981–2010. ncwa -a time takes their arithmetic mean and removes the time dimension. The query shows one grid-point value.

In [ ]:
ncwa -a time -d time,5,359,12 nco/baseline.nc nco/clim_06.nc

**Inspect the result**

In [ ]:
ncks -H -C -v tas -d lat,0 -d lon,0 nco/clim_06.nc

### 05.13 · Calculate the July reference

NCO indices start at zero. The slice time,6,359,12 selects indices 6, 18, 30, …, 354: the 30 July fields in 1981–2010. ncwa -a time takes their arithmetic mean and removes the time dimension. The query shows one grid-point value.

In [ ]:
ncwa -a time -d time,6,359,12 nco/baseline.nc nco/clim_07.nc

**Inspect the result**

In [ ]:
ncks -H -C -v tas -d lat,0 -d lon,0 nco/clim_07.nc

### 05.14 · Calculate the August reference

NCO indices start at zero. The slice time,7,359,12 selects indices 7, 19, 31, …, 355: the 30 August fields in 1981–2010. ncwa -a time takes their arithmetic mean and removes the time dimension. The query shows one grid-point value.

In [ ]:
ncwa -a time -d time,7,359,12 nco/baseline.nc nco/clim_08.nc

**Inspect the result**

In [ ]:
ncks -H -C -v tas -d lat,0 -d lon,0 nco/clim_08.nc

### 05.15 · Calculate the September reference

NCO indices start at zero. The slice time,8,359,12 selects indices 8, 20, 32, …, 356: the 30 September fields in 1981–2010. ncwa -a time takes their arithmetic mean and removes the time dimension. The query shows one grid-point value.

In [ ]:
ncwa -a time -d time,8,359,12 nco/baseline.nc nco/clim_09.nc

**Inspect the result**

In [ ]:
ncks -H -C -v tas -d lat,0 -d lon,0 nco/clim_09.nc

### 05.16 · Calculate the October reference

NCO indices start at zero. The slice time,9,359,12 selects indices 9, 21, 33, …, 357: the 30 October fields in 1981–2010. ncwa -a time takes their arithmetic mean and removes the time dimension. The query shows one grid-point value.

In [ ]:
ncwa -a time -d time,9,359,12 nco/baseline.nc nco/clim_10.nc

**Inspect the result**

In [ ]:
ncks -H -C -v tas -d lat,0 -d lon,0 nco/clim_10.nc

### 05.17 · Calculate the November reference

NCO indices start at zero. The slice time,10,359,12 selects indices 10, 22, 34, …, 358: the 30 November fields in 1981–2010. ncwa -a time takes their arithmetic mean and removes the time dimension. The query shows one grid-point value.

In [ ]:
ncwa -a time -d time,10,359,12 nco/baseline.nc nco/clim_11.nc

**Inspect the result**

In [ ]:
ncks -H -C -v tas -d lat,0 -d lon,0 nco/clim_11.nc

### 05.18 · Calculate the December reference

NCO indices start at zero. The slice time,11,359,12 selects indices 11, 23, 35, …, 359: the 30 December fields in 1981–2010. ncwa -a time takes their arithmetic mean and removes the time dimension. The query shows one grid-point value.

In [ ]:
ncwa -a time -d time,11,359,12 nco/baseline.nc nco/clim_12.nc

**Inspect the result**

In [ ]:
ncks -H -C -v tas -d lat,0 -d lon,0 nco/clim_12.nc

### 05.19 · Combine the twelve reference fields

ncecat -u month creates a month dimension. The explicit file order runs from January through December; the result has twelve reference fields.

In [ ]:
ncecat -u month nco/clim_01.nc nco/clim_02.nc nco/clim_03.nc nco/clim_04.nc nco/clim_05.nc nco/clim_06.nc nco/clim_07.nc nco/clim_08.nc nco/clim_09.nc nco/clim_10.nc nco/clim_11.nc nco/clim_12.nc nco/climatology_unlabelled.nc

**Inspect the result**

In [ ]:
ncks -m -C -v tas nco/climatology_unlabelled.nc

### 05.20 · Label calendar months 1–12

Assign the twelve calendar-month labels explicitly and save climatology.nc. The query prints all twelve labels.

In [ ]:
ncap2 -s 'month[month]={1,2,3,4,5,6,7,8,9,10,11,12};month@long_name="calendar month";' nco/climatology_unlabelled.nc nco/climatology.nc

**Inspect the result**

In [ ]:
ncks -H -C -v month nco/climatology.nc

### 05.21 · Name the reference temperature

ncrename -v old,new renames tas to tas_clim in a new file, so the monthly reference can coexist with the original tas.

In [ ]:
ncrename -v tas,tas_clim nco/climatology.nc nco/clim_named.nc

**Inspect the result**

In [ ]:
ncks -m -C -v tas_clim nco/clim_named.nc

### 05.22 · Fix the month dimension

Fix the twelve-month dimension before appending the reference. The inherited classic-model format permits one record dimension; time remains that dimension in the temperature file.

In [ ]:
ncks --fix_rec_dmn month nco/clim_named.nc nco/clim_fixed.nc

**Inspect the result**

In [ ]:
ncks -m -C -v tas_clim nco/clim_fixed.nc

### 05.23 · Append the reference fields

ncks -A appends the selected tas_clim variable to celsius.nc. The monthly temperature values remain unchanged. The query shows the appended reference dimensions.

In [ ]:
ncks -A -v tas_clim nco/clim_fixed.nc nco/celsius.nc

**Inspect the result**

In [ ]:
ncks -m -C -v tas_clim nco/celsius.nc

### 05.24 · Subtract the matching monthly reference

The 552 monthly records start in January. Within the ncap2 expression, t%12 selects the corresponding zero-based reference month. Each anomaly is tas minus tas_clim at the same latitude and longitude. The query shows the first anomaly at one grid point.

In [ ]:
ncap2 -s 'tas_anom=tas;for(t=0;t<552;t++){tas_anom(t,:,:)=tas(t,:,:)-tas_clim(t%12,:,:);}tas_anom@units="degC";tas_anom@anomaly_reference_period="1981-2010";' nco/celsius.nc nco/anomaly_all.nc

**Inspect the result**

In [ ]:
ncks -H -C -v tas_anom -d time,0 -d lat,0 -d lon,0 nco/anomaly_all.nc

### 05.25 · Save the monthly anomaly fields

Select tas_anom; its coordinates and time bounds are retained automatically in anomaly.nc. Its metadata shows 552 monthly fields; the twelve reference fields are no longer part of this output.

In [ ]:
ncks -v tas_anom nco/anomaly_all.nc nco/anomaly.nc

**Inspect the result**

In [ ]:
ncks -m -C -v tas_anom nco/anomaly.nc

### 05.26 · Calculate each monthly duration

Subtract the lower time bound from the upper bound. duration is measured in the file’s time units. The query prints the first twelve monthly weights.

In [ ]:
ncap2 -s 'duration[time]=time_bnds(:,1)-time_bnds(:,0);' nco/anomaly.nc nco/weighted.nc

**Inspect the result**

In [ ]:
ncks -H -C -v duration -d time,0,11 nco/weighted.nc

### 05.27 · Calculate the duration-weighted mean map

ncwa -a time -w duration computes Σ(anomaly × duration) / Σ(duration) at every grid point. map_mean.nc is a period-mean anomaly, not a trend. The query prints one map value.

In [ ]:
ncwa -a time -w duration -v tas_anom nco/weighted.nc nco/map_mean.nc

**Inspect the result**

In [ ]:
ncks -H -C -v tas_anom -d lat,0 -d lon,0 nco/map_mean.nc

### 05.28 · Select the Arctic and average longitudes

Select latitude centres from 65°N to 90°N and use ncwa -a lon to average all longitudes equally. The query shows one latitude from the first monthly zonal-mean field.

In [ ]:
ncwa -a lon -d lat,65.0,90.0 nco/anomaly.nc nco/arctic_zonal.nc

**Inspect the result**

In [ ]:
ncks -H -C -v tas_anom -d time,0 -d lat,0 nco/arctic_zonal.nc

### 05.29 · Average the Arctic latitudes

ncwa -a lat gives equal weight to the remaining latitude centres. No area weights are supplied. arctic_monthly.nc contains 552 monthly Arctic anomalies; the query shows its first twelve values.

In [ ]:
ncwa -a lat nco/arctic_zonal.nc nco/arctic_monthly.nc

**Inspect the result**

In [ ]:
ncks -H -C -v tas_anom -d time,0,11 nco/arctic_monthly.nc

### 05.30 · Average twelve months per year

ncra --mro retains multiple output records. The last two numbers in -d time,,,12,12 specify a twelve-record stride and a twelve-record averaging group. The result contains 46 annual values, 1979–2024, with equal monthly weights. The query shows the first five years.

In [ ]:
ncra --mro -d time,,,12,12 nco/arctic_monthly.nc nco/arctic_annual.nc

**Inspect the result**

In [ ]:
ncks -H -C -v tas_anom -d time,0,4 nco/arctic_annual.nc

### 05.31 · Select the January bounds

Select monthly records 0, 12, 24, …, 540: January of each year. Their lower bounds provide the starts of the 46 annual intervals. The query shows the first pair of monthly bounds in the file’s numeric time units.

In [ ]:
ncks -v time_bnds -d time,0,,12 nco/arctic_monthly.nc nco/year_starts.nc

**Inspect the result**

In [ ]:
ncks -H -C -v time_bnds -d time,0 nco/year_starts.nc

### 05.32 · Select the December bounds

Select records 11, 23, 35, …, 551: December of each year. Their upper bounds provide the ends of the 46 annual intervals.

In [ ]:
ncks -v time_bnds -d time,11,,12 nco/arctic_monthly.nc nco/year_ends.nc

**Inspect the result**

In [ ]:
ncks -H -C -v time_bnds -d time,0 nco/year_ends.nc

### 05.33 · Extract the end of each year

The second column of the December bounds is the first day of the following January. Store it as year_end; the query prints its first three values.

In [ ]:
ncap2 -s 'year_end[time]=time_bnds(:,1);' nco/year_ends.nc nco/year_ends_named.nc

**Inspect the result**

In [ ]:
ncks -H -C -v year_end -d time,0,2 nco/year_ends_named.nc

### 05.34 · Append the year-end values

Append year_end to the January-bound file. -C copies only the requested variable, preserving the existing time coordinate.

In [ ]:
ncks -A -C -v year_end nco/year_ends_named.nc nco/year_starts.nc

**Inspect the result**

In [ ]:
ncks -H -C -v year_end -d time,0,2 nco/year_starts.nc

### 05.35 · Construct complete annual bounds

Retain each January lower bound and replace the upper bound with year_end. The first output interval now spans one complete calendar year.

In [ ]:
ncap2 -s 'time_bnds(:,1)=year_end;' nco/year_starts.nc nco/year_bounds.nc

**Inspect the result**

In [ ]:
ncks -H -C -v time_bnds -d time,0 nco/year_bounds.nc

### 05.36 · Update the annual bounds

ncra averages monthly bounds as ordinary variables. Replace those bounds with the complete annual intervals just constructed. -A -C appends only time_bnds; the annual anomaly values and time coordinate are preserved.

In [ ]:
ncks -A -C -v time_bnds nco/year_bounds.nc nco/arctic_annual.nc

**Inspect the result**

In [ ]:
ncks -H -C -v time_bnds -d time,0 nco/arctic_annual.nc

## 06 · Compare the results

Check **five products** from each command-line route against Python: monthly
climatology, monthly anomaly fields, mean anomaly map, Arctic monthly series and
annual series. These ten comparisons also check coordinates, the reference-period
residual and the annual definition.

In [ ]:
# Compare the three results
subprocess.run([sys.executable, str(ROOT / "scripts/compare_updated_workflows.py"), str(RUN)], check=True)
report = json.loads((RUN / "updated_validation.json").read_text())
assert report["passed"]
print("Maximum absolute difference (°C):", max(row["max_absolute_difference_degC"] for row in report["comparisons"].values()))
print("Ten product comparisons passed; inspect the report for coordinate, baseline and annual checks.")

## 07 · Read the figures

Read each row across Python, CDO and NCO. The maps and stripes use shared color
limits. The first row is a period-mean anomaly, not a trend. The inputs are one
model realization under historical forcing followed by SSP5-8.5, not observations.

These checks compare the three implementations. A direct comparison with
Evgenia's saved ESMValTool outputs is a separate check.

In [ ]:
# Show the comparison figure
display(Image(filename=str(RUN / "updated_comparison.png")))

## 08 · Try a different region or period

Open [04_monthly_anomaly_hands_on.ipynb](04_monthly_anomaly_hands_on.ipynb).
Change the Arctic boundary or annual time weights, one at a time.

Each run folder contains NetCDF outputs, `updated_validation.json` and
`updated_comparison.png`. Download your results and edited notebook before
closing a Binder session.